# Demo of Spark GraphFrames

1. Khởi tạo Spark Session với GraphFrames và Hive

In [1]:
import os, sys
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, concat

spark = SparkSession.builder \
    .appName("Credit_Card_Fraud_Graph_Processing") \
    .config("spark.jars.packages", "graphframes:graphframes:0.8.2-spark3.2-s_2.12") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .config("spark.sql.warehouse.dir", "hdfs://namenode:9000/user/hive/warehouse") \
    .enableHiveSupport() \
    .getOrCreate()



2. Đọc dữ liệu Parquet từ Hive (hoặc HDFS)

In [2]:
# Đọc dữ liệu từ Hive table (giả sử bảng tên là 'credit_transactionscleaned_transactionscredit_transaction_db.cleaned_transactions')
# Hoặc đọc trực tiếp từ thư mục Parquet trên HDFS
spark.sql("USE credit_transaction_db")
df = spark.table("cleaned_transactions")

# Hiển thị Schema và số lượng dòng để kiểm tra
df.printSchema()
print(f"Tổng số giao dịch: {df.count():,}")

root
 |-- TRANSACTION_ID: long (nullable = true)
 |-- TX_DATETIME: timestamp (nullable = true)
 |-- CUSTOMER_ID: long (nullable = true)
 |-- TERMINAL_ID: long (nullable = true)
 |-- TX_AMOUNT: double (nullable = true)
 |-- TX_TIME_SECONDS: long (nullable = true)
 |-- TX_TIME_DAYS: integer (nullable = true)
 |-- TX_FRAUD: integer (nullable = true)
 |-- TX_FRAUD_SCENARIO: integer (nullable = true)
 |-- tx_date: date (nullable = true)
 |-- tx_hour: integer (nullable = true)
 |-- tx_day_of_week: string (nullable = true)
 |-- is_weekend: integer (nullable = true)
 |-- is_night: integer (nullable = true)
 |-- customer_tx_count: long (nullable = true)
 |-- customer_avg_amount: double (nullable = true)
 |-- terminal_tx_count: long (nullable = true)
 |-- tx_year_month: string (nullable = true)

Tổng số giao dịch: 14,158,985


3. Định nghĩa Bảng Đỉnh (Vertices)
Trong cấu trúc Đồ thị hai phía (Bipartite Graph), Vertex bắt buộc phải có cột tên là id. Ta sẽ tạo Đỉnh cho cả Khách hàng và Máy POS.

In [3]:
from pyspark.sql.functions import concat

# Thêm prefix C_ cho Khách hàng
customers = df.select(concat(lit("C_"), col("CUSTOMER_ID").cast("string")).alias("id")) \
              .distinct() \
              .withColumn("type", lit("Customer"))

# Thêm prefix T_ cho Terminal
terminals = df.select(concat(lit("T_"), col("TERMINAL_ID").cast("string")).alias("id")) \
              .distinct() \
              .withColumn("type", lit("Terminal"))

# Gộp lại thành bảng Vertices duy nhất
vertices = customers.union(terminals)

print(f"Tổng số đỉnh (Vertices): {vertices.count():,}")
vertices.show(5)

Tổng số đỉnh (Vertices): 109,997
+------+--------+
|    id|    type|
+------+--------+
|C_4756|Customer|
|C_3628|Customer|
|C_4672|Customer|
|C_2605|Customer|
|C_5013|Customer|
+------+--------+
only showing top 5 rows



4. Định nghĩa Bảng Cạnh (Edges)
Cạnh đại diện cho giao dịch. Theo quy ước của GraphFrames, nguồn gửi phải là cột src và đích đến phải là cột dst. Luồng giao dịch sẽ đi từ CUSTOMER_ID -> TERMINAL_ID.

In [4]:
# Cập nhật ID nguồn và đích khớp với định dạng mới ở Vertices
edges = df.withColumn("src", concat(lit("C_"), col("CUSTOMER_ID").cast("string"))) \
          .withColumn("dst", concat(lit("T_"), col("TERMINAL_ID").cast("string")))

# Các cột còn lại (TX_DATETIME, TX_AMOUNT, TX_FRAUD...) tự động trở thành thuộc tính của Cạnh
print(f"Tổng số cạnh (Edges): {edges.count():,}")
edges.show(5)

Tổng số cạnh (Edges): 14,158,985
+--------------+-------------------+-----------+-----------+---------+---------------+------------+--------+-----------------+----------+-------+--------------+----------+--------+-----------------+-------------------+-----------------+-------------+------+-------+
|TRANSACTION_ID|        TX_DATETIME|CUSTOMER_ID|TERMINAL_ID|TX_AMOUNT|TX_TIME_SECONDS|TX_TIME_DAYS|TX_FRAUD|TX_FRAUD_SCENARIO|   tx_date|tx_hour|tx_day_of_week|is_weekend|is_night|customer_tx_count|customer_avg_amount|terminal_tx_count|tx_year_month|   src|    dst|
+--------------+-------------------+-----------+-----------+---------+---------------+------------+--------+-----------------+----------+-------+--------------+----------+--------+-----------------+-------------------+-----------------+-------------+------+-------+
|       6724753|2019-03-13 14:49:57|       9661|      99899|     4.28|       29947797|         346|       0|                0|2019-03-13|     14|           Wed|         

5. Khởi tạo GraphFrame và Phân tích cơ bản

In [5]:
from graphframes import GraphFrame

# Khởi tạo Mạng lưới Đồ thị
g = GraphFrame(vertices, edges)

# Bật checkpoint directory (Bắt buộc với một số thuật toán đồ thị nặng)
spark.sparkContext.setCheckpointDir("hdfs://namenode:9000/user/spark/checkpoints")


/usr/local/spark/python/pyspark/sql/dataframe.py:168: UserWarning: DataFrame.sql_ctx is an internal property, and will be removed in future releases. Use DataFrame.sparkSession instead.
  warnings.warn(


In [6]:
# Demo 1: In ra các Motif cơ bản (Đã tối ưu chọn cột để tránh OOM)
print("Demo Motif cơ bản:")
motifs = g.find("(a)-[e]->(b)").select("a.id", "e.TX_AMOUNT", "e.TX_DATETIME", "b.id")
motifs.show(5, truncate=False)

Demo Motif cơ bản:


/usr/local/spark/python/pyspark/sql/dataframe.py:147: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


+------+---------+-------------------+-------+
|id    |TX_AMOUNT|TX_DATETIME        |id     |
+------+---------+-------------------+-------+
|C_4756|137.2    |2019-03-06 04:13:35|T_99899|
|C_3628|72.85    |2019-03-03 08:06:07|T_66788|
|C_4672|31.22    |2019-03-12 07:27:14|T_67694|
|C_2605|121.62   |2019-03-17 00:52:22|T_1613 |
|C_2605|10.73    |2019-03-01 09:06:59|T_67876|
+------+---------+-------------------+-------+
only showing top 5 rows



In [7]:
# Demo 2: Tìm In-Degree (Số lượng khách hàng đến quẹt thẻ tại mỗi Terminal)
print("Top 5 Máy POS có lượng giao dịch đến (In-degree) cao nhất:")
inDegreeDF = g.inDegrees
inDegreeDF.orderBy(col("inDegree").desc()).show(5)

Top 5 Máy POS có lượng giao dịch đến (In-degree) cao nhất:
+-------+--------+
|     id|inDegree|
+-------+--------+
|T_11678|     254|
|T_74295|     251|
|T_55218|     249|
|T_42247|     248|
|T_43006|     246|
+-------+--------+
only showing top 5 rows

